# Air Quality Small Language Model (SLM) — Complete Training Pipeline

**Project:** Domain-Specific SLM for Air Quality Data  
**Architecture:** Qwen3 (RMSNorm + Grouped-Query Attention + RoPE + SwiGLU)  
**Dataset:** UCI Machine Learning Repository — Air Quality Dataset  
**Reference:** Professor's TinyStories training notebook (adapted for Air Quality domain)  

---

## Pipeline Overview
1. Environment & GPU check
2. Install dependencies
3. Load and inspect the Air Quality dataset
4. Prepare and tokenize the training data
5. Define the Qwen3-based SLM architecture
6. Configure and run training
7. Evaluate on validation and test sets
8. Save model, tokenizer, and configuration
9. Run inference on Air Quality prompts
10. Display training summary

> **Note:** For long training runs, prefer the command-line script:  
> `python src/train_slm.py`

## Cell 1 — Install Dependencies

In [ ]:
# Install required packages (run once)
import subprocess, sys
pkgs = ['torch', 'transformers', 'datasets', 'tiktoken', 'pandas', 'numpy', 'tqdm']
for pkg in pkgs:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All dependencies ready.')

## Cell 2 — Environment & GPU Check

In [ ]:
import sys, os, torch
from pathlib import Path

print('=' * 60)
print('AIR QUALITY SLM — ENVIRONMENT CHECK')
print('=' * 60)
print(f'Python     : {sys.version.split()[0]}')
print(f'PyTorch    : {torch.__version__}')
print(f'CUDA       : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    DEVICE = 'cuda'
else:
    print('GPU        : Not available — training will run on CPU')
    DEVICE = 'cpu'

# Project paths
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # SLM/
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print(f'Device     : {DEVICE.upper()}')
print(f'Project    : {PROJECT_ROOT}')
print('=' * 60)

## Cell 3 — Load and Inspect the Air Quality Dataset

In [ ]:
import pandas as pd

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df   = pd.read_csv(PROCESSED_DIR / 'validation.csv')
test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')

print('Air Quality Dataset Splits:')
print(f'  Train      : {len(train_df):,} rows')
print(f'  Validation : {len(val_df):,} rows')
print(f'  Test       : {len(test_df):,} rows')
print(f'  Columns    : {list(train_df.columns)}')
print()
print('Sample train row:')
display(train_df.head(2))

## Cell 4 — Load Tokenizer (Qwen2.5, matching professor's reference)

In [ ]:
from transformers import AutoTokenizer

print('Loading Qwen tokenizer (matches professor reference — Qwen2.5/Qwen3)...')
try:
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B', trust_remote_code=True)
    TOK_ID = 'Qwen/Qwen2.5-0.5B'
except:
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-8B', trust_remote_code=True)
    TOK_ID = 'Qwen/Qwen3-8B'

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

VOCAB_SIZE = max(len(tokenizer), tokenizer.vocab_size)
print(f'Tokenizer  : {TOK_ID}')
print(f'Vocab size : {VOCAB_SIZE:,}')
print(f'EOS token  : {repr(tokenizer.eos_token)} (id={tokenizer.eos_token_id})')

# Quick tokenization test
test_str = 'Air quality measurement: CO(GT)=1.5 mg/m3, NOx(GT)=89 ppb, Temperature=15.5 C'
tokens = tokenizer.encode(test_str)
print(f'\nSample encode: "{test_str[:50]}..." -> {len(tokens)} tokens')

## Cell 5 — Prepare Tokenized Binary Data (matches professor's Cell 4 approach)

In [ ]:
# This replicates the professor's tokenization loop from TinyStories Cell 4,
# adapted to our Air Quality instruction dataset.
from prepare_training_data import prepare_splits

BIN_DIR = PROJECT_ROOT / 'training' / 'data_bin'

if (BIN_DIR / 'train.bin').exists() and (BIN_DIR / 'val.bin').exists():
    print('Pre-tokenized binary files already exist:')
    for split in ['train', 'val', 'test']:
        p = BIN_DIR / f'{split}.bin'
        if p.exists():
            size_mb = p.stat().st_size / 1e6
            print(f'  {split}.bin : {size_mb:.1f} MB')
else:
    print('Creating tokenized binary files...')
    meta = prepare_splits(tokenizer)
    print('Done:', meta)

## Cell 6 — Import Model Architecture from train_slm.py

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Import the Qwen3-based model defined in src/train_slm.py
from train_slm import (
    Qwen3Model,
    get_data_memmap,
    get_batch,
    estimate_loss,
    BIN_DIR,
    CHECKPOINT_DIR,
    FINAL_MODEL_DIR
)

print('Qwen3Model imported successfully.')
print(f'BIN_DIR        : {BIN_DIR}')
print(f'CHECKPOINT_DIR : {CHECKPOINT_DIR}')
print(f'FINAL_MODEL_DIR: {FINAL_MODEL_DIR}')

## Cell 7 — Configure Model & Hyperparameters (auto-scales to CPU or GPU)

In [ ]:
import time
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

torch.manual_seed(42)
np.random.seed(42)

if DEVICE == 'cuda':
    d_model, n_heads, n_kv_groups = 256, 8, 4
    n_layers, d_ff, block_size    = 6, 1024, 128
    batch_size, grad_accum        = 16, 4
    max_iters, lr                 = 1500, 4e-4
    eval_interval, eval_iters     = 250, 50
else:  # CPU
    d_model, n_heads, n_kv_groups = 128, 4, 2
    n_layers, d_ff, block_size    = 4, 512, 64
    batch_size, grad_accum        = 8, 1
    max_iters, lr                 = 500, 1e-3
    eval_interval, eval_iters     = 50, 15

config = dict(
    vocab_size=VOCAB_SIZE, d_model=d_model, n_heads=n_heads,
    n_kv_groups=n_kv_groups, n_layers=n_layers, d_ff=d_ff,
    max_seq_len=block_size, tie_weights=True
)

model = Qwen3Model(**config).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())

print('Model Configuration:')
print(f'  Architecture : Qwen3 (RMSNorm + GQA + RoPE + SwiGLU)')
print(f'  Vocab Size   : {VOCAB_SIZE:,}')
print(f'  Hidden Dim   : {d_model}')
print(f'  Layers       : {n_layers}')
print(f'  Heads / KV   : {n_heads} / {n_kv_groups}')
print(f'  Block Size   : {block_size}')
print(f'  Total Params : {total_params:,} (~{total_params/1e6:.1f}M)')
print(f'  Batch Size   : {batch_size} x grad_accum={grad_accum} = {batch_size*grad_accum} effective')
print(f'  Max Iters    : {max_iters}')
print(f'  Device       : {DEVICE.upper()}')

## Cell 8 — Load Tokenized Data & Sanity Check (verifies shapes and loss)

In [ ]:
print('Loading tokenized binary memmap files...')
memmap_cache = {
    'train': get_data_memmap('train'),
    'val'  : get_data_memmap('val'),
    'test' : get_data_memmap('test'),
}

print(f'Train tokens : {len(memmap_cache["train"]):,}')
print(f'Val tokens   : {len(memmap_cache["val"]):,}')
print(f'Test tokens  : {len(memmap_cache["test"]):,}')

# SANITY CHECK — verify batch shapes and forward pass
print('\nRunning sanity check...')
X, Y = get_batch('train', memmap_cache, block_size, batch_size, DEVICE)
print(f'  Batch X shape : {X.shape}  (batch_size x block_size)')
print(f'  Batch Y shape : {Y.shape}')
assert X.shape == (batch_size, block_size), 'Shape mismatch!'

with torch.no_grad():
    logits, loss = model(X, Y)
print(f'  Logits shape  : {logits.shape}  (batch x seq x vocab)')
print(f'  Sanity loss   : {loss.item():.4f} (expected ~{math.log(VOCAB_SIZE):.1f} for random init)')
print('Sanity check PASSED.')

## Cell 9 — Training Loop (matches professor's approach from TinyStories Cell 4–5)

In [ ]:
from tqdm.auto import tqdm
from contextlib import nullcontext
import json

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
best_ckpt = CHECKPOINT_DIR / 'best_model.pt'
last_ckpt = CHECKPOINT_DIR / 'latest_model.pt'

optimizer = torch.optim.AdamW(
    model.parameters(), lr=lr, betas=(0.9, 0.95),
    weight_decay=0.1, eps=1e-9
)
warmup_steps   = max(20, int(max_iters * 0.1))
sched_warm     = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_steps)
sched_cos      = CosineAnnealingLR(optimizer, T_max=max_iters - warmup_steps, eta_min=lr * 0.1)
scheduler      = SequentialLR(optimizer, schedulers=[sched_warm, sched_cos], milestones=[warmup_steps])

use_amp   = (DEVICE == 'cuda')
scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)
ctx       = torch.amp.autocast(device_type=DEVICE, dtype=torch.float16) if use_amp else nullcontext()

# Baseline loss before training
init = estimate_loss(model, memmap_cache, block_size, batch_size, DEVICE, eval_iters=10)
print(f'Baseline — Train: {init["train"]:.4f}  Val: {init["val"]:.4f}')

best_val = float('inf')
loss_history = []
t0 = time.time()

pbar = tqdm(range(max_iters), desc='Training')
for step in pbar:
    if (step > 0 and step % eval_interval == 0) or step == max_iters - 1:
        losses = estimate_loss(model, memmap_cache, block_size, batch_size, DEVICE, eval_iters)
        clr    = optimizer.param_groups[0]['lr']
        loss_history.append({'step': step, **losses, 'lr': clr})
        pbar.write(f'Step {step:4d}/{max_iters} | Train: {losses["train"]:.4f} | Val: {losses["val"]:.4f} | PPL: {math.exp(min(losses["val"],20)):.2f}')
        torch.save({'step': step, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'config': config, 'losses': losses}, last_ckpt)
        if losses['val'] < best_val:
            best_val = losses['val']
            torch.save({'step': step, 'model_state_dict': model.state_dict(),
                        'config': config, 'best_val_loss': best_val}, best_ckpt)

    optimizer.zero_grad(set_to_none=True)
    acc_loss = 0.0
    for _ in range(grad_accum):
        X, Y = get_batch('train', memmap_cache, block_size, batch_size, DEVICE)
        with ctx:
            _, loss = model(X, Y)
            sl = loss / grad_accum
        (scaler.scale(sl) if use_amp else sl).backward()
        acc_loss += sl.item()
    if use_amp:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        scaler.step(optimizer); scaler.update()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
    scheduler.step()
    pbar.set_postfix({'loss': f'{acc_loss:.4f}', 'best_val': f'{best_val:.4f}'})

elapsed = time.time() - t0
print(f'\nTraining done in {elapsed/60:.1f} min. Best val loss: {best_val:.4f} (PPL={math.exp(min(best_val,20)):.2f})')

## Cell 10 — Evaluate Best Model on Validation & Test Sets

In [ ]:
# Load best checkpoint
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded best checkpoint from step {ckpt["step"]}.')

# Validation evaluation
val_losses = torch.zeros(20)
for k in range(20):
    X, Y = get_batch('val', memmap_cache, block_size, batch_size, DEVICE)
    with torch.no_grad():
        _, l = model(X, Y)
    val_losses[k] = l.item()
final_val_loss = float(val_losses.mean())

# Test evaluation
test_losses = torch.zeros(20)
for k in range(20):
    X, Y = get_batch('test', memmap_cache, block_size, batch_size, DEVICE)
    with torch.no_grad():
        _, l = model(X, Y)
    test_losses[k] = l.item()
final_test_loss = float(test_losses.mean())

print(f'Final Validation Loss : {final_val_loss:.4f}  (PPL = {math.exp(min(final_val_loss,20)):.2f})')
print(f'Final Test Loss       : {final_test_loss:.4f}  (PPL = {math.exp(min(final_test_loss,20)):.2f})')

## Cell 11 — Save Model, Tokenizer & Config to models/air_quality_slm/

In [ ]:
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save tokenizer
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f'Tokenizer saved to {FINAL_MODEL_DIR}')

# Save weights
torch.save(model.state_dict(), FINAL_MODEL_DIR / 'pytorch_model.bin')
print('Model weights saved: pytorch_model.bin')

# Save HF-compatible config
hf_config = {
    'model_type': 'air_quality_slm',
    'architectures': ['Qwen3ForCausalLM'],
    'vocab_size': config['vocab_size'],
    'hidden_size': config['d_model'],
    'num_attention_heads': config['n_heads'],
    'num_key_value_heads': config['n_kv_groups'],
    'num_hidden_layers': config['n_layers'],
    'intermediate_size': config['d_ff'],
    'max_position_embeddings': config['max_seq_len'],
    'tie_word_embeddings': config['tie_weights'],
    'rms_norm_eps': 1e-6,
    'rope_theta': 10000.0,
    'torch_dtype': 'float32',
    'transformers_version': '5.23.0',
}
with open(FINAL_MODEL_DIR / 'config.json', 'w') as f:
    json.dump(hf_config, f, indent=2)
print('config.json saved.')

# Save training summary
summary = {
    'model_name': 'Air Quality Small Language Model (SLM)',
    'base_architecture': 'Qwen3 (GQA, RoPE, RMSNorm, SwiGLU)',
    'dataset': 'UCI Machine Learning Repository - Air Quality Dataset',
    'parameters': config,
    'total_params': total_params,
    'training_time_seconds': round(elapsed, 2),
    'iterations': max_iters,
    'initial_train_loss': round(init['train'], 4),
    'initial_val_loss': round(init['val'], 4),
    'best_val_loss': round(best_val, 4),
    'best_val_perplexity': round(math.exp(min(best_val, 20)), 2),
    'final_val_loss': round(final_val_loss, 4),
    'final_test_loss': round(final_test_loss, 4),
    'final_test_perplexity': round(math.exp(min(final_test_loss, 20)), 2),
    'loss_progression': loss_history,
    'status': 'TRAINED_AND_VERIFIED'
}
with open(FINAL_MODEL_DIR / 'training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('training_summary.json saved.')

## Cell 12 — Inference Demo: Air Quality Prompts

In [ ]:
sample_prompts = [
    'Task: Predict the absolute humidity given the following parameters: Date: 11/03/2004, Time: 02.00.00, CO(GT): 0.9, NOx(GT): 45.0, NO2(GT): 60.0. Response:',
    'Air quality measurement report for 15/04/2004 at 12.00.00: Carbon Monoxide CO(GT)=',
    'Question: What were the ambient temperature and relative humidity recorded on 10/03/2004 at 18.00.00? Answer:',
    'Query: What are the primary pollutant concentrations (CO, NOx, NO2) on 12/03/2004 at 08.00.00? Result:',
]

print('=' * 65)
print('       AIR QUALITY SLM — INFERENCE GENERATION RESULTS')
print('=' * 65)

generation_results = []
model.eval()
for i, prompt in enumerate(sample_prompts, 1):
    input_ids = torch.tensor(
        tokenizer.encode(prompt, add_special_tokens=False),
        dtype=torch.long
    ).unsqueeze(0).to(DEVICE)
    output_ids = model.generate(input_ids, max_new_tokens=50, temperature=0.7, top_k=40)
    full = tokenizer.decode(output_ids.squeeze().tolist(), skip_special_tokens=False)
    completion = full[len(prompt):].strip()
    print(f'\n[Sample {i}]')
    print(f'Prompt     : {prompt[:80]}...')
    print(f'Completion : {completion}')
    generation_results.append({'prompt': prompt, 'completion': completion})

with open(FINAL_MODEL_DIR / 'sample_generations.json', 'w') as f:
    json.dump(generation_results, f, indent=2)
print('\n' + '=' * 65)

## Cell 13 — Training Summary

In [ ]:
print('\n' + '=' * 65)
print('       AIR QUALITY SLM — TRAINING COMPLETE SUMMARY')
print('=' * 65)
print(f'Model         : Qwen3-based SLM ({total_params/1e6:.1f}M params)')
print(f'Dataset       : UCI Air Quality — {len(train_df)+len(val_df)+len(test_df):,} rows total')
print(f'Train/Val/Test: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}')
print(f'Training Time : {elapsed/60:.1f} minutes')
print(f'Iterations    : {max_iters}')
print()
print(f'Initial Train Loss : {init["train"]:.4f}')
print(f'Final Train Loss   : {loss_history[-1]["train"]:.4f}')
print(f'Best Val Loss      : {best_val:.4f}  (PPL = {math.exp(min(best_val, 20)):.2f})')
print(f'Test Loss          : {final_test_loss:.4f}  (PPL = {math.exp(min(final_test_loss, 20)):.2f})')
print()
print(f'Saved model    : {FINAL_MODEL_DIR}')
print(f'Best checkpoint: {best_ckpt}')
print()
print('Ready for Hugging Face upload!')
print('=' * 65)